# Notebook 00 — Colab Smoke Test

Run this **once, before the real experiments**. It exercises the four things that have never been executed anywhere:

1. `build_transforms()` — albumentations is not installable on the local Windows machine, so the whole online pipeline is unexecuted code
2. CUDA — every check so far ran on CPU
3. `num_workers > 0` — every check so far used 0
4. a full epoch at 512×512 through transforms, model, loss and metrics

It also measures how much slower Drive is than the session disk, which is the only way to decide whether copying the data is worth it.

Ten minutes here is cheaper than discovering a problem in the fourth run of six, where it would cost the comparability of everything already done.

# 0. Environment

Mounts Drive and reports the GPU. If no GPU is reported, stop: change the runtime type before going further.

In [ ]:
# =====================================================
# 0.1 - BOOTSTRAP
# =====================================================
import sys
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/VineProject_ToF")
LOCAL_ROOT = Path("/content/VineProject_ToF")

if not Path("/content").exists():
    raise RuntimeError("This notebook is meant to run on Colab.")

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = LOCAL_ROOT if (LOCAL_ROOT / "src" / "config.py").exists() else DRIVE_ROOT
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import config

config.print_environment_info()

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime > Change runtime type > GPU, then run again.")

In [ ]:
# =====================================================
# 0.2 - IS ALBUMENTATIONS THERE, AND WHICH VERSION
# =====================================================
# The transforms use `masks` (plural) and run on a 5-channel image. Both are
# supported in principle, but this is the one function of the project that has
# never been executed, so its version matters.
import subprocess

try:
    import albumentations
except ImportError:
    print("albumentations missing — installing")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "albumentations"], check=True
    )
    import albumentations

print(f"albumentations {albumentations.__version__}")

# 1. Drive Against the Session Disk

The `Dataset` opens **five files per sample**: three modalities, the label map and the skeleton. Over 70 epochs that is more than a hundred thousand reads.

Drive is mounted through FUSE and is slow on small files; `/content` is a local disk. This cell measures the difference on the actual data instead of assuming it.

Read the ratio, not the absolute numbers: what matters is whether reading is comparable to the GPU step or an order of magnitude above it.

In [ ]:
# =====================================================
# 1.1 - MEASURE THE READ COST
# =====================================================
import shutil
import time

import cv2

SAMPLE_COUNT = 40


def time_reads(root: Path, label: str) -> float:
    """Read the same files the Dataset would, and time it."""
    folder = root / "data" / "augmented" / "train" / "intensity"
    files = sorted(folder.glob("*.png"))[:SAMPLE_COUNT]
    if not files:
        print(f"  {label}: no files at {folder}")
        return float("nan")

    started = time.perf_counter()
    for path in files:
        cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    elapsed = time.perf_counter() - started

    print(f"  {label:<16} {len(files)} files in {elapsed:.2f}s "
          f"({1000 * elapsed / len(files):.1f} ms/file)")
    return elapsed / len(files)


print("Cold read from Drive:")
drive_cost = time_reads(DRIVE_ROOT, "Drive")

In [ ]:
# =====================================================
# 1.2 - COPY TO THE SESSION DISK AND MEASURE AGAIN
# =====================================================
# Drive stays the canonical copy; this is a working copy that disappears with
# the session. Results have to be written back before it ends — see cell 5.2.
if not (LOCAL_ROOT / "src" / "config.py").exists():
    started = time.perf_counter()
    shutil.copytree(DRIVE_ROOT, LOCAL_ROOT, dirs_exist_ok=True)
    print(f"Copied in {time.perf_counter() - started:.0f}s")
else:
    print(f"{LOCAL_ROOT} already present")

local_cost = time_reads(LOCAL_ROOT, "session disk")

if drive_cost == drive_cost and local_cost == local_cost:
    print()
    print(f"Drive is {drive_cost / local_cost:.1f}x slower per file.")
    print("Per epoch that is roughly "
          f"{(drive_cost - local_cost) * 5 * 295:.0f}s of extra reading, "
          "before any overlap from the loader workers.")

# 2. Restart Here

If the copy was made, **restart the runtime now** (Runtime → Restart session) and run cell 0.1 again. Python caches imported modules, so `config` is still the one loaded from Drive and every path derived from it still points there.

After the restart, cell 0.1 will report the session disk as the project root.

# 3. The Online Pipeline

The first execution of `build_transforms()` anywhere. Then one batch is pushed through it, which is what verifies that a 5-channel image and a list of masks both survive the geometry.

The check that matters is the last one: the label must move with the image. A transform applied to one and not the other would leave the labels describing pixels that have shifted, and no metric would report it.

In [ ]:
# =====================================================
# 3.1 - BUILD THE TRANSFORMS
# =====================================================
from data_preparation import get_splits_hash
from dataset import build_transforms, create_dataloaders

SPLITS_HASH = get_splits_hash()
print(f"Split: {SPLITS_HASH}")

TRANSFORMS = build_transforms(splits_hash=SPLITS_HASH)
print(f"[OK] transforms built: {list(TRANSFORMS)}")

In [ ]:
# =====================================================
# 3.2 - ONE BATCH THROUGH IT
# =====================================================
DATALOADERS = create_dataloaders(transforms=TRANSFORMS, num_workers=0, verbose=True)

images, labels, skeletons = next(iter(DATALOADERS["train"]))
height, width = config.IMG_SIZE

print()
print(f"  images    {tuple(images.shape)}  {images.dtype}")
print(f"  labels    {tuple(labels.shape)}  {labels.dtype}")
print(f"  skeletons {tuple(skeletons.shape)}  {skeletons.dtype}")

assert images.shape[1] == config.INPUT_CHANNELS, "wrong channel count"
assert tuple(images.shape[2:]) == (height, width), "wrong spatial size"
assert labels.shape == skeletons.shape == (images.shape[0], height, width)
assert labels.max() < config.NUM_CLASSES, "label outside the declared classes"
print("\n[OK] shapes and label range are as declared")

In [ ]:
# =====================================================
# 3.3 - DID THE LABELS MOVE WITH THE IMAGE
# =====================================================
# The failure this catches is silent: geometry applied to the image but not to
# its masks. It would show as a model that never learns, weeks later.
from exploration import show_augmented_batch

show_augmented_batch(DATALOADERS["train"], n_samples=3)

# 4. The Model on GPU

`test_model` runs a forward **and** a backward pass at the configured batch size and image size, and reports the peak memory. The backward is where memory peaks: a model that forwards fine can still exhaust the GPU once gradients are allocated.

If the peak is close to the card's capacity, lower `config.BATCH_SIZE` **now** — every run in the comparison must share it, so changing it later invalidates whatever came before.

In [ ]:
# =====================================================
# 4.1 - EVERY ARCHITECTURE AT THE REAL BATCH SIZE
# =====================================================
from models import create_model, test_model

for architecture in ("UNet", "SegNet", "PSPNet"):
    model = create_model(architecture).to(config.DEVICE)
    ok = test_model(model)
    del model
    torch.cuda.empty_cache()
    if not ok:
        raise RuntimeError(f"{architecture} failed its check")

# 5. A Short Run With Everything On

Two epochs with `num_workers` as configured, the skeleton loss active, clDice on validation, checkpointing and the results row. This is the whole pipeline, only shorter.

The skeleton loss is switched on deliberately: it is the path with the most moving parts, so if anything breaks it should break here.

In [ ]:
# =====================================================
# 5.1 - TWO EPOCHS
# =====================================================
from losses import create_loss_function, create_skeleton_loss
from training import train_model

config.SKELETON_LOSS_CONFIG["enabled"] = True     # exercise the harder path

DATALOADERS = create_dataloaders(transforms=TRANSFORMS, verbose=True)   # workers as configured

MODEL = create_model().to(config.DEVICE)
CRITERION = create_loss_function(
    class_weights=config.load_class_weights(expected_splits_hash=SPLITS_HASH)
).to(config.DEVICE)

TRACKER, BEST = train_model(
    model=MODEL,
    dataloaders=DATALOADERS,
    criterion=CRITERION,
    skeleton_criterion=create_skeleton_loss(),
    run_id="SMOKE_TEST",
    num_epochs=2,
    checkpoint_interval=1,
)

In [ ]:
# =====================================================
# 5.2 - WHAT WAS WRITTEN, AND THE RESUME PATH
# =====================================================
import json

from training import find_resume_point

root = config.EXPERIMENTS_DIR / "SMOKE_TEST"
for path in sorted(root.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(root)}  {path.stat().st_size / 1024:.0f} KB")

history = json.loads((root / "history.json").read_text(encoding="utf-8"))
print()
print(f"  epochs recorded: {history['epochs_run']}   completed: {history['completed']}")
print(f"  metrics per epoch: {len(history['history']['val'][0])}")
print(f"  resume point: {find_resume_point('SMOKE_TEST')}")

In [ ]:
# =====================================================
# 5.3 - THE RESUME ACTUALLY CONTINUES
# =====================================================
# Two more epochs from the checkpoint. The history must reach 4 without a gap.
TRACKER, BEST = train_model(
    model=MODEL,
    dataloaders=DATALOADERS,
    criterion=CRITERION,
    skeleton_criterion=create_skeleton_loss(),
    run_id="SMOKE_TEST",
    num_epochs=4,
    resume_from=find_resume_point("SMOKE_TEST"),
    checkpoint_interval=1,
)

print()
print(f"  epochs in history: {len(TRACKER.history['val'])} (expected 4)")
assert len(TRACKER.history["val"]) == 4, "the resume did not continue the history"
print("[OK] the run continued as though it had not stopped")

# 6. Clean Up and Decide

The smoke-test run is removed: it is two epochs on a model that learned nothing, and leaving it would put a meaningless row in the comparison table.

Then the decision. If the session disk was clearly faster in cell 1.2, the real runs should work from `/content` — and **results must be copied back to Drive before the session ends**, because `/content` does not survive it.

In [ ]:
# =====================================================
# 6.1 - REMOVE THE SMOKE-TEST RUN
# =====================================================
smoke = config.EXPERIMENTS_DIR / "SMOKE_TEST"
if smoke.exists():
    shutil.rmtree(smoke)
    print(f"removed {smoke}")

config.SKELETON_LOSS_CONFIG["enabled"] = False    # back to the configured default

In [ ]:
# =====================================================
# 6.2 - COPY RESULTS BACK TO DRIVE
# =====================================================
# Run this at the end of every session that trained from /content. Only the
# experiment folders and the results table need to travel back: the data and
# the code came from Drive and are unchanged.
def sync_results_to_drive() -> None:
    """Copy experiments/ and results/ from the session disk to Drive."""
    if PROJECT_ROOT == DRIVE_ROOT:
        print("Already working on Drive, nothing to copy.")
        return

    for folder in ("experiments", "results"):
        source = PROJECT_ROOT / folder
        if not source.exists():
            continue
        destination = DRIVE_ROOT / folder
        shutil.copytree(source, destination, dirs_exist_ok=True)
        print(f"  {folder} -> {destination}")


sync_results_to_drive()